# Heart Disease Prediction System
## Exploratory Data Analysis and Preprocessing Notebook

Project:
Heart Disease Prediction System using End-to-End Machine Learning Pipeline

Dataset:
Heart Disease Dataset (Kaggle)

Objectives:
- Understand dataset characteristics
- Validate data quality
- Perform exploratory data analysis
- Prepare dataset for machine learning
- Generate reproducible preprocessing workflow
- Produce processed datasets for model training

## Cell 2 (Import Libraries)

python3 -m venv .venv  
source .venv/bin/activate

pip install -U numpy pandas matplotlib seaborn scipy scikit-learn joblib jupyter notebook

pip freeze > requirements.txt

pip install -r requirements.txt

In [5]:
import sys
print(sys.executable)
print(sys.version)

/shared/Projects/Eksperimen_SML/.venv/bin/python
3.14.4 (main, Apr  8 2026, 04:02:31) [GCC 15.2.0]


In [4]:
import warnings
warnings.filterwarnings("ignore")

import os
import json
import random
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import joblib

ModuleNotFoundError: No module named 'seaborn'

## Cell 3 (Configuration)

In [ ]:
RANDOM_STATE = 42

TEST_SIZE = 0.15
VALIDATION_SIZE = 0.15

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", None)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## Cell 4 (Directory Configuration)

In [ ]:
RAW_DATA_PATH = "../data/raw/heart.csv"

INTERIM_DIR = "../data/interim"
PROCESSED_DIR = "../data/processed"
REPORT_DIR = "../reports"
ARTIFACT_DIR = "../artifacts"

os.makedirs(INTERIM_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(ARTIFACT_DIR, exist_ok=True)

## Cell 5 (Data Loading)

In [ ]:
df = pd.read_csv(RAW_DATA_PATH)

print("Dataset Shape:", df.shape)
df.head()

## Cell 6 (Initial Exploration)

In [ ]:
df.info()
df.describe().T
df.sample(5)

## Cell 7 (Data Validation)

In [ ]:
expected_columns = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal",
    "target"
]

missing_columns = set(expected_columns) - set(df.columns)
additional_columns = set(df.columns) - set(expected_columns)

validation_report = {
    "dataset_shape": df.shape,
    "missing_columns": list(missing_columns),
    "additional_columns": list(additional_columns),
    "duplicate_columns": int(df.columns.duplicated().sum())
}

validation_report

with open(
    f"{REPORT_DIR}/data_validation.json",
    "w"
) as f:
    json.dump(validation_report, f, indent=4)

## Cell 8 (Missing Value Analysis)

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (
        df.isnull().sum()/len(df)
    )*100
})

missing.sort_values(
    by="missing_count",
    ascending=False
)

missing.to_json(
    f"{REPORT_DIR}/missing_value_report.json",
    indent=4
)

## Cell 9 (Duplicate Analysis)

In [ ]:
duplicates = df.duplicated().sum()

print(f"Duplicate rows: {duplicates}")

duplicate_report = {
    "duplicate_rows": int(duplicates)
}

with open(
    f"{REPORT_DIR}/duplicate_report.json",
    "w"
) as f:
    json.dump(
        duplicate_report,
        f,
        indent=4
    )

df = df.drop_duplicates()

print(df.shape)

## Cell 10 (Univariate Analysis)

In [ ]:
numerical_features = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak"
]

for col in numerical_features:
    fig, ax = plt.subplots(
        1,
        2,
        figsize=(14,4)
    )

    sns.histplot(
        df[col],
        kde=True,
        ax=ax[0]
    )

    sns.boxplot(
        x=df[col],
        ax=ax[1]
    )

    plt.show()

# categorial

categorical_features = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal",
    "target"
]

for col in categorical_features:
    plt.figure(figsize=(6,4))
    sns.countplot(
        data=df,
        x=col
    )
    plt.show()

In [ ]:
for col in categorical_features[:-1]:
    plt.figure(figsize=(7,5))

    sns.countplot(
        data=df,
        x=col,
        hue="target"
    )

    plt.show()

In [ ]:
for col in numerical_features:
    plt.figure(figsize=(8,5))

    sns.boxplot(
        data=df,
        x="target",
        y=col
    )

    plt.show()

In [ ]:
outlier_report = {}

for col in numerical_features:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - (1.5 * IQR)
    upper = Q3 + (1.5 * IQR)

    outliers = df[
        (df[col] < lower)
        |
        (df[col] > upper)
    ]

    outlier_report[col] = len(outliers)

outlier_report

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(12,8))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.show()

In [ ]:
df["age_group"] = pd.cut(
    df["age"],
    bins=[0,40,55,100],
    labels=[
        "young",
        "middle",
        "old"
    ]
)

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

categorical_columns = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal"
]

numerical_columns = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak"
]

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numerical_columns
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_columns
        )
    ]
)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

In [ ]:
X_train_processed = preprocessor.fit_transform(
    X_train
)

X_val_processed = preprocessor.transform(
    X_val
)

X_test_processed = preprocessor.transform(
    X_test
)

In [ ]:
feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed.toarray(),
    columns=feature_names
)

X_val_processed = pd.DataFrame(
    X_val_processed.toarray(),
    columns=feature_names
)

X_test_processed = pd.DataFrame(
    X_test_processed.toarray(),
    columns=feature_names
)

In [ ]:
X_train_processed.to_csv(
    f"{PROCESSED_DIR}/X_train.csv",
    index=False
)

X_val_processed.to_csv(
    f"{PROCESSED_DIR}/X_val.csv",
    index=False
)

X_test_processed.to_csv(
    f"{PROCESSED_DIR}/X_test.csv",
    index=False
)

y_train.to_csv(
    f"{PROCESSED_DIR}/y_train.csv",
    index=False
)

y_val.to_csv(
    f"{PROCESSED_DIR}/y_val.csv",
    index=False
)

y_test.to_csv(
    f"{PROCESSED_DIR}/y_test.csv",
    index=False
)

In [ ]:
df.to_csv(
    f"{INTERIM_DIR}/heart_cleaned.csv",
    index=False
)

In [ ]:
joblib.dump(
    preprocessor,
    f"{ARTIFACT_DIR}/preprocessor.pkl"
)

In [ ]:
print("Train :", X_train_processed.shape)
print("Validation :", X_val_processed.shape)
print("Test :", X_test_processed.shape)

# Conclusions

1. Dataset contains 1025 observations and 13 features.
2. Duplicate records were removed.
3. Missing values are minimal/non-existent.
4. Several numerical variables contain outliers but remain medically meaningful.
5. Data preprocessing pipeline has been created successfully.
6. Processed datasets and preprocessing artifacts have been generated.
7. The dataset is ready for the modelling phase.